In [12]:
from sklearn.datasets import load_breast_cancer
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1. Load built-in medical dataset directly from Python
cancer_data = load_breast_cancer()
X = pd.DataFrame(cancer_data.data, columns=cancer_data.feature_names)
y = cancer_data.target

# 2. Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Create scaled versions for Tasks 2, 3, and 4
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Setup Complete! Built-in data is successfully loaded.")

Setup Complete! Built-in data is successfully loaded.


In [13]:
# Task 1: Re-training with Unscaled Data
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

# Using the original (NOT scaled) datasets as instructed
# Replace 'best_k' with the numerical value you found earlier (e.g., 5)
k_value = 5 
knn_unscaled = KNeighborsClassifier(n_neighbors=k_value)

# Fit the model using original unscaled training data
knn_unscaled.fit(X_train, y_train)

# Predict using the original unscaled test data
y_pred_unscaled = knn_unscaled.predict(X_test)

# Calculate accuracy
accuracy_unscaled = accuracy_score(y_test, y_pred_unscaled)
print(f"Accuracy with Unscaled Data: {accuracy_unscaled}")

Accuracy with Unscaled Data: 0.956140350877193


Analysis for Task 1: > When using unscaled data, the model's accuracy typically drops compared to the scaled version. This happens because KNN is a distance-based algorithm relying on Euclidean calculations. Without normalization, features with much larger numerical ranges completely dominate the distance calculations, causing the algorithm to ignore features with smaller ranges.

In [15]:
# Task 2: Tuning Weights and Distance
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import f1_score

# Setting weights to distance and using p=1 for Manhattan distance
best_k = 5
knn_tuned = KNeighborsClassifier(n_neighbors=best_k, weights='distance', p=1)
knn_tuned.fit(X_train_scaled, y_train)

y_pred_tuned = knn_tuned.predict(X_test_scaled)
print(f"Tuned F1-Score: {f1_score(y_test, y_pred_tuned)}")

Tuned F1-Score: 0.971830985915493


Analysis for Task 2: > By setting weights='distance', data points closer to the query sample have a stronger influence on the classification decision. Changing the metric to $p=1$ applies Manhattan Distance, which is often more robust against dataset outliers than default Euclidean distance. Review the printed F1-Score to determine if these settings optimized your overall performance.  

In [16]:
# Task 3: Clinical Priority - Minimizing False Negatives
from sklearn.metrics import confusion_matrix, recall_score

# We test different K values to see which one is "safest" for medical use
for k in [3, 5, 7, 9, 11]:
    knn_medical = KNeighborsClassifier(n_neighbors=k)
    knn_medical.fit(X_train_scaled, y_train)
    y_pred_medical = knn_medical.predict(X_test_scaled)
    
    # Extract False Negatives (Sick patients the model called "Healthy")
    # In the confusion matrix, this is at index [1][0]
    fn = confusion_matrix(y_test, y_pred_medical)[1][0]
    recall = recall_score(y_test, y_pred_medical)
    
    print(f"K={k} | Missed Cases (False Negatives): {fn} | Recall: {recall:.4f}")

K=3 | Missed Cases (False Negatives): 3 | Recall: 0.9577
K=5 | Missed Cases (False Negatives): 3 | Recall: 0.9577
K=7 | Missed Cases (False Negatives): 3 | Recall: 0.9577
K=9 | Missed Cases (False Negatives): 2 | Recall: 0.9718
K=11 | Missed Cases (False Negatives): 2 | Recall: 0.9718


Analysis for Task 3: >
 In a diagnostic medical environment like cancer detection, a False Negative is significantly more critical and dangerous than a False Positive because it means a patient's life-saving treatment could be delayed. 
 Therefore, we must prioritize Recall over Precision to ensure the system catches as many positive cases as possible.

In [17]:
# Task 4: Training with Only 2 Features
from sklearn.neighbors import KNeighborsClassifier

# Using the best K we found (9) but only the first two columns of data
knn_simple = KNeighborsClassifier(n_neighbors=9)

# We slice the data to use only the first 2 features [:, :2]
knn_simple.fit(X_train_scaled[:, :2], y_train)
accuracy_simple = knn_simple.score(X_test_scaled[:, :2], y_test)

print(f"Accuracy with only 2 features: {accuracy_simple}")

Accuracy with only 2 features: 0.9298245614035088


Analysis for Task 4: > Restricting the model input to only the first two features drops the model's accuracy. While limiting the model to two dimensions simplifies visualization, it strips away too many critical clinical features, making a 2-feature configuration too simple and unsafe for deployment in a real-world medical setting.